# 03 · QLoRA pilot (M2) and a three-way comparison

The same 500 training examples, hyperparameters and 500 validation questions as the LoRA pilot in notebook 02. The only
change is the method (`configs/qlora.yaml`): the language model is loaded in 4-bit NF4 with double quantization, the vision
tower stays in 16-bit, and the optimizer is paged AdamW 8-bit.

This notebook also re-scores M0 and the LoRA pilot (M1, the adapter attached from notebook 02's output) with per-prediction
token counts. All three models can then be compared on accuracy, format, latency **and** generated tokens per second.

Validation data only. Settings: Accelerator → GPU T4, Internet → On, Input → notebook output
`visiontune-02-baseline-lora-pilot`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess, sys

REPO_DIR = "/tmp/VisionTune"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/akhilnathreddy/VisionTune.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
COMMIT = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(COMMIT)

In [ ]:
!pip install -q -U "transformers>=4.57" "peft>=0.17" "accelerate>=1.0" "datasets>=3.0" "bitsandbytes>=0.45" qwen-vl-utils
!pip install -q -e .

In [ ]:
import glob, json, time
from pathlib import Path

OUT = Path("/kaggle/working/run03")
(OUT / "predictions").mkdir(parents=True, exist_ok=True)
(OUT / "outputs").mkdir(parents=True, exist_ok=True)
N_VAL = 500
timings = {}

def sh(cmd):
    print("$", cmd, flush=True)
    t0 = time.perf_counter()
    subprocess.run(cmd, shell=True, check=True)
    return round(time.perf_counter() - t0, 1)

matches = glob.glob("/kaggle/input/**/scienceqa_qwen25vl3b_lora_r16_seed42/adapter/adapter_config.json", recursive=True)
LORA_ADAPTER = os.path.dirname(matches[0]) if matches else None
print("LoRA pilot adapter from notebook 02:", LORA_ADAPTER)

## 1. Data

In [ ]:
timings["prepare_s"] = sh("python -m visiontune.data.prepare --config configs/base.yaml --skip-ai2d > /dev/null")

## 2. QLoRA pilot training (M2)

In [ ]:
timings["qlora_train_s"] = sh(f"python -m visiontune.training.train --config configs/qlora.yaml train.output_root={OUT}/outputs")
run_dir = OUT / "outputs/scienceqa_qwen25vl3b_qlora_r16_seed42"
qlora = json.loads((run_dir / "run_summary.json").read_text())
print(json.dumps({k: qlora[k] for k in ("train_examples", "train_seconds_this_session", "peak_vram_gb", "adapter_size_mb",
                                        "lora_targets", "parameters", "linear_layers", "final_eval")}, indent=1))
print("train loss:", [round(h["loss"], 3) for h in qlora["log_history"] if "loss" in h])

## 3. Score M0, M1 and M2 on the same 500 validation questions

In [ ]:
preds = {
    "base": OUT / "predictions" / f"base_scienceqa_validation_n{N_VAL}.jsonl",
    "qlora_pilot": OUT / "predictions" / f"qlora_pilot_scienceqa_validation_n{N_VAL}.jsonl",
}
timings["qlora_eval_s"] = sh(f"python -m visiontune.inference.predict --config configs/qlora.yaml --adapter {run_dir}/adapter "
                             f"--split validation --limit {N_VAL} --out {preds['qlora_pilot']}")
timings["base_eval_s"] = sh(f"python -m visiontune.inference.predict --config configs/base.yaml "
                            f"--split validation --limit {N_VAL} --out {preds['base']}")
if LORA_ADAPTER:
    preds["lora_pilot"] = OUT / "predictions" / f"lora_pilot_scienceqa_validation_n{N_VAL}.jsonl"
    timings["lora_eval_s"] = sh(f"python -m visiontune.inference.predict --config configs/lora.yaml --adapter {LORA_ADAPTER} "
                                f"--split validation --limit {N_VAL} --out {preds['lora_pilot']}")

In [ ]:
from itertools import combinations
from visiontune.data.io import read_jsonl
from visiontune.evaluation.metrics import mcnemar_exact, paired_bootstrap

metrics = {k: json.loads(p.with_suffix(".metrics.json").read_text()) for k, p in preds.items()}
correct = {k: {r["id"]: r["correct"] for r in read_jsonl(p)} for k, p in preds.items()}
ids = sorted(correct["base"])
assert all(sorted(c) == ids for c in correct.values()), "all models must be scored on identical examples"

summary = {k: {"accuracy": m["overall"], "format": m["format"],
               "near_dup": m["by_near_dup_in_train"],
               "latency_s_per_example_batched": m.get("latency_s_per_example_batched"),
               "mean_new_tokens": m.get("mean_new_tokens"),
               "generated_tokens_per_s_batched": m.get("generated_tokens_per_s_batched"),
               "peak_vram_gb": m.get("peak_vram_gb")} for k, m in metrics.items()}
pairs = {}
for a, b in combinations(preds, 2):
    xa, xb = [correct[a][i] for i in ids], [correct[b][i] for i in ids]
    pairs[f"{b}_minus_{a}"] = {"paired_bootstrap": paired_bootstrap(xa, xb), "mcnemar": mcnemar_exact(xa, xb)}

result = {"n": len(ids), "models": summary, "pairs": pairs, "timings_s": timings, "commit": COMMIT,
          "qlora_run": {k: qlora[k] for k in ("train_seconds_this_session", "peak_vram_gb", "adapter_size_mb",
                                               "lora_targets", "parameters", "final_eval")}}
(OUT / "comparison_m0_m1_m2.json").write_text(json.dumps(result, indent=2))
print(json.dumps({k: {"acc": round(v["accuracy"]["accuracy"], 4), "strict": v["format"]["strict"],
                      "tok": v["mean_new_tokens"], "tok/s": v["generated_tokens_per_s_batched"],
                      "s/ex": v["latency_s_per_example_batched"], "vram": v["peak_vram_gb"]} for k, v in summary.items()}, indent=1))
print(json.dumps({k: {"diff": round(v["paired_bootstrap"]["diff"], 4), "p": v["paired_bootstrap"]["p_value"],
                      "mcnemar": v["mcnemar"]} for k, v in pairs.items()}, indent=1))

In [ ]:
import shutil
for ckpt in (OUT / "outputs").glob("*/checkpoint-*"):
    shutil.rmtree(ckpt)
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.relative_to(OUT)}")